<a href="https://colab.research.google.com/github/RadRebelSam/ai4all-20c-darkpatterns/blob/main/AI4ALL_20C_Dark_Pattern_Recognition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dark Pattern Recognition: Full Colab Workflow

This notebook trains and evaluates the current AI4ALL Ignite Team 20C dark-pattern project.

It includes the current two-model workflow:

- **Model 1:** binary `Dark Pattern` vs `Not Dark Pattern` detector.
- **Model 2:** second-stage category/type classifier for suspicious text.

It also includes demo filters, updated visualizations, example text analysis, and a lightweight webpage scanner. Run the cells from top to bottom. CPU runtime is enough; no GPU is required.

## 0. Start Fresh in Colab

This removes an older cloned copy if it exists, then clones the latest GitHub repo.

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/RadRebelSam/ai4all-20c-darkpatterns.git"
PROJECT_DIR = Path("/content/ai4all-20c-darkpatterns")

# Important: leave the project folder before deleting it. Otherwise Colab can
# lose its current working directory and git clone may fail.
os.chdir("/content")

if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)
    print("Removed old repo folder")

subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)

os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))
print("Current folder:", Path.cwd())


## 1. Install Packages

In [ ]:
!pip install -q -r requirements.txt seaborn beautifulsoup4 lxml requests openpyxl

## 2. Check Dataset Files

In [ ]:
from pathlib import Path

expected_files = [
    "krishuppal - dark-patterns.csv",
    "devitachi - dark-pattern.csv",
    "- pattern_classifications.csv",
    "adarshm09 - dark-pattern-dataset.csv",
    "mohitsharma527 - dark-patterns-on-ecommerce-platforms.csv",
]

for filename in expected_files:
    path = Path("datasets") / filename
    print(("FOUND   " if path.exists() else "MISSING ") + str(path))

## 3. Load Binary and Category Datasets

The project uses:

- a balanced binary dataset for Model 1
- a compatible dark-pattern-only category dataset for Model 2
- an expanded binary dataset for robustness checking

In [ ]:
import pandas as pd

from src.data import (
    TEXT_COLUMN,
    LABEL_COLUMN,
    CATEGORY_COLUMN,
    SOURCE_COLUMN,
    load_primary_binary_dataset,
    load_expanded_binary_dataset,
    load_dark_pattern_category_dataset,
    summarize_labels,
    validate_binary_dataset,
    validate_category_dataset,
)

primary_df = load_primary_binary_dataset()
expanded_df = load_expanded_binary_dataset()
category_df = load_dark_pattern_category_dataset()

validate_binary_dataset(primary_df)
validate_binary_dataset(expanded_df)
validate_category_dataset(category_df)

print("Primary binary rows:", len(primary_df))
print("Primary label counts:", summarize_labels(primary_df))
print()
print("Expanded binary rows:", len(expanded_df))
print("Expanded label counts:", summarize_labels(expanded_df))
print()
print("Second-stage category rows:", len(category_df))
print("Category counts:")
print(category_df[CATEGORY_COLUMN].value_counts().sort_index())

display(primary_df.head())

## 4. Visualize Dataset Balance

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

primary_df[LABEL_COLUMN].map({0: "Not Dark Pattern", 1: "Dark Pattern"}).value_counts().plot(
    kind="bar",
    ax=axes[0],
    color=["#b8b8b8", "#ff8a65"],
)
axes[0].set_title("Primary Dataset Class Balance")
axes[0].set_xlabel("")
axes[0].set_ylabel("Rows")

category_df[CATEGORY_COLUMN].value_counts().sort_values().plot(
    kind="barh",
    ax=axes[1],
    color="#86c5a9",
)
axes[1].set_title("Second-Stage Category Dataset")
axes[1].set_xlabel("Rows")
axes[1].set_ylabel("")

plt.tight_layout()
plt.show()

## 5. Train and Compare Model 1 and Model 2

Both stages compare the same five classifiers:

- Logistic Regression
- Naive Bayes
- Linear SVM
- Decision Tree
- Random Forest

In [ ]:
from src.modeling import (
    train_and_compare,
    train_and_compare_categories,
    results_to_frame,
    category_results_to_frame,
)

primary_results = train_and_compare(primary_df)
primary_metrics = results_to_frame(primary_results)

expanded_results = train_and_compare(expanded_df)
expanded_metrics = results_to_frame(expanded_results)

category_results = train_and_compare_categories(category_df)
category_metrics = category_results_to_frame(category_results)

best_primary = primary_results[0]
best_category = category_results[0]

print("Model 1: primary binary dataset results")
display(primary_metrics)
print("Expanded binary dataset experiment")
display(expanded_metrics)
print("Model 2: second-stage category model results")
display(category_metrics)

print("Best Model 1:", best_primary.name)
print("Best Model 2:", best_category.name)

## 6. Visualize Model Comparisons

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.barplot(data=primary_metrics, x="f1", y="model", ax=axes[0], color="#ff6b6b")
axes[0].set_title("Model 1: Binary Detection F1")
axes[0].set_xlabel("F1")
axes[0].set_ylabel("")
axes[0].set_xlim(0, 1)

sns.barplot(data=category_metrics, x="f1_macro", y="model", ax=axes[1], color="#4dabf7")
axes[1].set_title("Model 2: Category Macro F1")
axes[1].set_xlabel("Macro F1")
axes[1].set_ylabel("")
axes[1].set_xlim(0, 1)

plt.tight_layout()
plt.show()

## 7. Two-Stage Pipeline Diagram

In [ ]:
from matplotlib.patches import FancyBboxPatch

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.axis("off")

boxes = [
    (0.04, 0.52, 0.22, 0.28, "Website text", "Snippet from app or webpage scan"),
    (0.34, 0.52, 0.24, 0.28, "Model 1", "Binary detector\nDark vs Not Dark"),
    (0.68, 0.62, 0.26, 0.22, "Not Dark", "Stop here"),
    (0.68, 0.28, 0.26, 0.24, "Model 2", "Likely type\nUrgency, Scarcity, etc."),
]
for x, y, w, h, title, body in boxes:
    patch = FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.03", facecolor="#e8f3ff", edgecolor="#263238")
    ax.add_patch(patch)
    ax.text(x + w/2, y + h*0.65, title, ha="center", va="center", fontsize=14, fontweight="bold")
    ax.text(x + w/2, y + h*0.35, body, ha="center", va="center", fontsize=10)

arrow = dict(arrowstyle="->", lw=2, color="#263238")
ax.annotate("", xy=(0.34, 0.66), xytext=(0.26, 0.66), arrowprops=arrow)
ax.annotate("", xy=(0.68, 0.73), xytext=(0.58, 0.67), arrowprops=arrow)
ax.annotate("", xy=(0.68, 0.41), xytext=(0.58, 0.58), arrowprops=arrow)
ax.text(0.60, 0.76, "if not suspicious", fontsize=9)
ax.text(0.60, 0.46, "if suspicious", fontsize=9)
plt.title("Two-Stage Dark Pattern Recognition Pipeline", fontsize=16, fontweight="bold")
plt.show()

## 8. Confusion Matrices and Per-Class Category F1

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix, precision_recall_fscore_support
from sklearn.model_selection import train_test_split

from src.modeling import make_pipeline, split_dataset

# Model 1 confusion matrix
x_train, x_test, y_train, y_test = split_dataset(primary_df)
default_pipeline = make_pipeline("Logistic Regression")
default_pipeline.fit(x_train, y_train)
y_pred = default_pipeline.predict(x_test)

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    display_labels=["Not Dark Pattern", "Dark Pattern"],
    cmap="Greens",
)
plt.title("Model 1 Confusion Matrix: Logistic Regression")
plt.show()

# Model 2 confusion matrix and per-class F1
x_cat_train, x_cat_test, y_cat_train, y_cat_test = train_test_split(
    category_df[TEXT_COLUMN],
    category_df[CATEGORY_COLUMN],
    test_size=0.2,
    stratify=category_df[CATEGORY_COLUMN],
    random_state=42,
)
category_pipeline = best_category.pipeline
cat_predictions = category_pipeline.predict(x_cat_test)
labels = sorted(y_cat_test.unique())

ConfusionMatrixDisplay.from_predictions(
    y_cat_test,
    cat_predictions,
    display_labels=labels,
    cmap="Blues",
    xticks_rotation=35,
)
plt.title("Model 2 Category Confusion Matrix")
plt.tight_layout()
plt.show()

precision, recall, f1, support = precision_recall_fscore_support(
    y_cat_test,
    cat_predictions,
    labels=labels,
    zero_division=0,
)
per_class = pd.DataFrame({
    "category": labels,
    "precision": precision,
    "recall": recall,
    "f1": f1,
    "support": support,
}).sort_values("f1")

display(per_class)

plt.figure(figsize=(10, 5.5))
sns.barplot(data=per_class, x="f1", y="category", color="#4ea5d9")
plt.title("Model 2 F1 by Dark Pattern Type")
plt.xlabel("F1 Score")
plt.ylabel("")
plt.xlim(0, 1.05)
plt.tight_layout()
plt.show()

## 9. Generate Project Figures

This runs the repository script that saves figures into both `reports/figures/` and `docs/figures/`.

In [ ]:
!python scripts/generate_visualizations.py

## 10. Predict Custom Text with Both Models

In [ ]:
examples = [
    "Only 2 left in stock. Buy now before it is gone.",
    "Free shipping on orders over $50.",
    "500 people are viewing this deal right now.",
    "Cotton t-shirt with crew neck and short sleeves.",
    "Limited time offer ends in 10 minutes.",
    "Previous price: $99.00 47% off",
]

probabilities = default_pipeline.predict_proba(examples)
predicted_labels = default_pipeline.predict(examples)

rows = []
for text, label, probs in zip(examples, predicted_labels, probabilities):
    row = {
        "text": text,
        "model_1_prediction": "Dark Pattern" if label == 1 else "Not Dark Pattern",
        "model_1_confidence": round(float(probs[label]), 4),
        "model_2_likely_type": "Not applicable",
    }
    if label == 1:
        row["model_2_likely_type"] = category_pipeline.predict([text])[0]
    rows.append(row)

display(pd.DataFrame(rows))

## 11. Demo Filters

These filters are demo safeguards. They do not retrain the model; they reduce obvious false positives from plain discounts, catalog titles, and low-context webpage fragments.

In [ ]:
from src.filters import (
    is_simple_price_or_discount_snippet,
    is_low_context_product_snippet,
    is_low_context_web_snippet,
    infer_dark_pattern_type,
)

filter_examples = [
    "Previous price: $99.00 47% off",
    "US stock 64GB touchscreen tablet refurbished open box",
    "Thanks Daniel for the walk through call yesterday.",
    "Hurry, 47% off ends soon!",
]

rows = []
for text in filter_examples:
    rows.append({
        "text": text,
        "plain_discount_filter": is_simple_price_or_discount_snippet(text),
        "product_title_filter": is_low_context_product_snippet(text),
        "low_context_web_filter": is_low_context_web_snippet(text),
        "rule_based_possible_type": infer_dark_pattern_type(text),
    })

display(pd.DataFrame(rows))

## 12. Lightweight Webpage Scanner for Colab

This Colab scanner uses `requests` and `BeautifulSoup`. It is lighter than Playwright and works well for many pages, but it may miss content rendered only after JavaScript runs.

In [ ]:
import requests
from bs4 import BeautifulSoup


def extract_visible_text_with_requests(url: str, timeout: int = 20) -> str:
    headers = {
        "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 Chrome/125 Safari/537.36"
    }
    response = requests.get(url, headers=headers, timeout=timeout)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "lxml")
    for tag in soup(["script", "style", "noscript", "svg", "iframe"]):
        tag.decompose()
    return soup.get_text("\n")


def split_visible_text(page_text: str, min_chars: int = 4, max_chars: int = 220) -> list[str]:
    parts = []
    for line in page_text.splitlines():
        cleaned = " ".join(line.split())
        if min_chars <= len(cleaned) <= max_chars:
            parts.append(cleaned)
    seen = set()
    unique = []
    for item in parts:
        key = item.lower()
        if key not in seen:
            seen.add(key)
            unique.append(item)
    return unique


def should_filter_web_snippet(text: str) -> bool:
    return (
        is_simple_price_or_discount_snippet(text)
        or is_low_context_product_snippet(text)
        or is_low_context_web_snippet(text)
    )


def scan_webpage(url: str, threshold: float = 0.65, limit: int = 10, apply_filters: bool = True):
    print("Scanning:", url)
    page_text = extract_visible_text_with_requests(url)
    snippets = split_visible_text(page_text)
    rows = []
    for snippet in snippets:
        if apply_filters and should_filter_web_snippet(snippet):
            continue
        label = int(default_pipeline.predict([snippet])[0])
        probs = default_pipeline.predict_proba([snippet])[0]
        confidence = float(probs[label])
        if label != 1 or confidence < threshold:
            continue
        rows.append({
            "snippet": snippet,
            "confidence": confidence,
            "likely_type": category_pipeline.predict([snippet])[0],
        })
    rows = sorted(rows, key=lambda row: row["confidence"], reverse=True)[:limit]
    print("Visible snippets reviewed:", len(snippets))
    print("Results shown:", len(rows))
    return pd.DataFrame(rows)

web_examples = [
    "https://www.ulta.com/promotion/sale",
    "https://toonbee.ai/",
    "https://www.cnn.com/",
]

scan_results = scan_webpage(web_examples[0], threshold=0.65, limit=10, apply_filters=True)
display(scan_results)

## 13. Optional Playwright Scanner

Use this if a page is JavaScript-heavy and the lightweight scanner misses important text. It is slower in Colab.

In [ ]:
# Optional setup:
# !pip install -q playwright
# !playwright install chromium

# Optional usage in Colab:
# from playwright.async_api import async_playwright
#
# async def extract_visible_text_with_playwright_colab(url: str, wait_ms: int = 3000) -> str:
#     async with async_playwright() as p:
#         browser = await p.chromium.launch(headless=True)
#         page = await browser.new_page(viewport={"width": 1365, "height": 900})
#         await page.goto(url, wait_until="domcontentloaded", timeout=45000)
#         await page.wait_for_timeout(wait_ms)
#         text = await page.locator("body").inner_text(timeout=10000)
#         await browser.close()
#         return text
#
# page_text = await extract_visible_text_with_playwright_colab("https://toonbee.ai/")
# snippets = split_visible_text(page_text)
# display(pd.DataFrame({"snippet": snippets[:20]}))

## 14. Custom Figure Generator

All project visualizations come from the model metrics, datasets, and trained/evaluated pipelines. The repository version lives in `scripts/generate_visualizations.py`.

Use the cells below if you want to regenerate better-looking figures directly in Colab. They save into both:

- `reports/figures/`
- `docs/figures/`

After running these cells, download the PNGs or commit the changed figure files to replace the current GitHub Page images.

In [ ]:
# Figure style knobs you can change.
FIG_DPI = 220
FIGURE_DIRS = [Path("reports") / "figures", Path("docs") / "figures"]
for figure_dir in FIGURE_DIRS:
    figure_dir.mkdir(parents=True, exist_ok=True)

plt.style.use("seaborn-v0_8-whitegrid")
PALETTE = sns.color_palette("Set2", 8)


def save_figure(fig, filename: str) -> None:
    """Save a figure to both report figures and GitHub Pages figures."""
    for figure_dir in FIGURE_DIRS:
        output_path = figure_dir / filename
        fig.savefig(output_path, dpi=FIG_DPI, bbox_inches="tight")
        print("Saved", output_path)


### 14A. Model Comparison Charts

In [ ]:
def plot_binary_model_comparison_custom(metrics: pd.DataFrame = primary_metrics):
    metric_cols = ["accuracy", "precision", "recall", "f1"]
    labels = ["Accuracy", "Precision", "Recall", "F1"]
    models = metrics["model"].tolist()
    y_positions = np.arange(len(models))
    bar_height = 0.18

    fig, ax = plt.subplots(figsize=(13, 7))
    for i, (col, label) in enumerate(zip(metric_cols, labels)):
        offsets = y_positions + i * bar_height
        values = metrics[col].tolist()
        bars = ax.barh(offsets, values, height=bar_height, label=label, color=PALETTE[i])
        for bar, value in zip(bars, values):
            ax.text(value + 0.004, bar.get_y() + bar.get_height() / 2, f"{value:.3f}", va="center", fontsize=10)

    ax.set_yticks(y_positions + bar_height * 1.5)
    ax.set_yticklabels(models, fontsize=11)
    ax.set_xlim(0.80, 1.03)
    ax.set_xlabel("Score", fontsize=12)
    ax.set_title("Model 1 Binary Classifier Comparison", fontsize=18, fontweight="bold")
    ax.legend(loc="lower right", frameon=True, fontsize=10)
    plt.tight_layout()
    save_figure(fig, "model_comparison.png")
    plt.show()


def plot_category_model_comparison_custom(metrics: pd.DataFrame = category_metrics):
    metric_cols = ["accuracy", "precision_macro", "recall_macro", "f1_macro"]
    labels = ["Accuracy", "Macro Precision", "Macro Recall", "Macro F1"]
    models = metrics["model"].tolist()
    y_positions = np.arange(len(models))
    bar_height = 0.18

    fig, ax = plt.subplots(figsize=(13, 7))
    for i, (col, label) in enumerate(zip(metric_cols, labels)):
        offsets = y_positions + i * bar_height
        values = metrics[col].tolist()
        bars = ax.barh(offsets, values, height=bar_height, label=label, color=PALETTE[i])
        for bar, value in zip(bars, values):
            ax.text(value + 0.004, bar.get_y() + bar.get_height() / 2, f"{value:.3f}", va="center", fontsize=10)

    ax.set_yticks(y_positions + bar_height * 1.5)
    ax.set_yticklabels(models, fontsize=11)
    ax.set_xlim(0.55, 1.03)
    ax.set_xlabel("Score", fontsize=12)
    ax.set_title("Model 2 Category Classifier Comparison", fontsize=18, fontweight="bold")
    ax.legend(loc="lower right", frameon=True, fontsize=10)
    plt.tight_layout()
    save_figure(fig, "category_model_comparison.png")
    plt.show()

plot_binary_model_comparison_custom()
plot_category_model_comparison_custom()


### 14B. Two-Model Pipeline and Summary Figures

In [ ]:
def plot_two_stage_pipeline_custom():
    fig, ax = plt.subplots(figsize=(15, 6))
    ax.axis("off")

    boxes = [
        (0.04, 0.57, 0.20, 0.22, "Website text", "Snippet from app\\nor webpage scan", PALETTE[2]),
        (0.31, 0.57, 0.22, 0.22, "Model 1", "Binary detector\\nDark vs Not Dark", PALETTE[0]),
        (0.62, 0.72, 0.28, 0.18, "Not Dark Pattern", "Stop: no type needed", PALETTE[7]),
        (0.62, 0.38, 0.28, 0.20, "Model 2", "Category classifier\\nfor suspicious text", PALETTE[1]),
        (0.36, 0.08, 0.54, 0.18, "Likely type", "Urgency, Scarcity, Social Proof, Misdirection, Obstruction, Sneaking, Forced Action", PALETTE[4]),
    ]

    for x, y, w, h, title, subtitle, color in boxes:
        patch = FancyBboxPatch(
            (x, y), w, h,
            boxstyle="round,pad=0.025,rounding_size=0.035",
            linewidth=1.6,
            edgecolor="#243042",
            facecolor=color,
            alpha=0.92,
        )
        ax.add_patch(patch)
        ax.text(x + w / 2, y + h * 0.65, title, ha="center", va="center", fontsize=17, fontweight="bold")
        ax.text(x + w / 2, y + h * 0.35, subtitle, ha="center", va="center", fontsize=11)

    arrow_kw = dict(arrowstyle="->", color="#243042", lw=2.4, shrinkA=5, shrinkB=5)
    ax.annotate("", xy=(0.31, 0.68), xytext=(0.24, 0.68), arrowprops=arrow_kw)
    ax.annotate("", xy=(0.62, 0.80), xytext=(0.53, 0.72), arrowprops=arrow_kw)
    ax.annotate("", xy=(0.62, 0.49), xytext=(0.53, 0.64), arrowprops=arrow_kw)
    ax.annotate("", xy=(0.63, 0.26), xytext=(0.76, 0.38), arrowprops=arrow_kw)
    ax.text(0.555, 0.83, "if not suspicious", fontsize=10, color="#475569")
    ax.text(0.545, 0.53, "if suspicious", fontsize=10, color="#475569")
    ax.set_title("Two-Stage Dark Pattern Recognition Pipeline", fontsize=20, fontweight="bold", pad=12)

    save_figure(fig, "two_stage_pipeline.png")
    plt.show()


def plot_two_model_summary_card_custom():
    fig, ax = plt.subplots(figsize=(13, 5.5))
    ax.axis("off")
    fig.patch.set_facecolor("#f8fafc")

    cards = [
        (0.06, "Model 1", "Binary detector", "TF-IDF + Logistic Regression", "F1 = 0.936", "Answers: Dark Pattern or Not Dark Pattern", PALETTE[0]),
        (0.54, "Model 2", "Type classifier", "TF-IDF + Linear SVM", "Macro F1 = 0.894", "Answers: likely dark-pattern category", PALETTE[1]),
    ]

    for x, label, title, model, score, note, color in cards:
        patch = FancyBboxPatch((x, 0.17), 0.40, 0.65, boxstyle="round,pad=0.025,rounding_size=0.035", linewidth=1.3, edgecolor="#cbd5e1", facecolor="white")
        ax.add_patch(patch)
        ax.text(x + 0.04, 0.69, label, fontsize=13, color="#64748b", fontweight="bold")
        ax.text(x + 0.04, 0.58, title, fontsize=22, color="#0f172a", fontweight="bold")
        ax.text(x + 0.04, 0.46, model, fontsize=13, color="#334155")
        ax.text(x + 0.04, 0.33, score, fontsize=26, color=color, fontweight="bold")
        ax.text(x + 0.04, 0.24, note, fontsize=12, color="#475569")

    ax.annotate("", xy=(0.53, 0.50), xytext=(0.46, 0.50), arrowprops=dict(arrowstyle="->", lw=2.2, color="#334155"))
    ax.text(0.50, 0.57, "if suspicious", ha="center", fontsize=10, color="#64748b")
    ax.set_title("Two Trained Models, Two Different Jobs", fontsize=20, fontweight="bold", pad=10)

    save_figure(fig, "two_model_summary_card.png")
    plt.show()

plot_two_stage_pipeline_custom()
plot_two_model_summary_card_custom()


### 14C. Confusion Matrices and Category F1 Chart

These are the charts most likely to need size/spacing tweaks. Adjust the `figsize`, font sizes, or text labels below if the GitHub Page looks crowded.

In [ ]:
from sklearn.metrics import confusion_matrix

def plot_binary_confusion_matrix_custom():
    cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
    class_labels = ["Not Dark Pattern", "Dark Pattern"]

    fig, ax = plt.subplots(figsize=(8, 7))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_labels, yticklabels=class_labels, linewidths=0.6, ax=ax, annot_kws={"fontsize": 13})
    ax.set_xlabel("Predicted", fontsize=13)
    ax.set_ylabel("Actual", fontsize=13)
    ax.set_title("Model 1 Confusion Matrix - Logistic Regression", fontsize=17, fontweight="bold")
    plt.tight_layout()
    save_figure(fig, "confusion_matrix.png")
    plt.show()


def plot_category_confusion_matrix_custom():
    labels = sorted(y_cat_test.unique())
    cm = confusion_matrix(y_cat_test, cat_predictions, labels=labels)

    fig, ax = plt.subplots(figsize=(13, 10))
    sns.heatmap(cm, annot=True, fmt="d", cmap="YlGnBu", xticklabels=labels, yticklabels=labels, linewidths=0.5, ax=ax, annot_kws={"fontsize": 11})
    ax.set_xlabel("Predicted", fontsize=14)
    ax.set_ylabel("Actual", fontsize=14)
    ax.set_title("Model 2 Category Confusion Matrix - Linear SVM", fontsize=18, fontweight="bold")
    ax.tick_params(axis="x", rotation=30, labelsize=11)
    ax.tick_params(axis="y", rotation=0, labelsize=11)
    plt.tight_layout()
    save_figure(fig, "category_confusion_matrix.png")
    plt.show()


def plot_category_per_class_f1_custom():
    scores = per_class.sort_values("f1")

    fig, ax = plt.subplots(figsize=(13, 8))
    bars = ax.barh(scores["category"], scores["f1"], color="#4ea5d9", height=0.62)
    for bar, value, count in zip(bars, scores["f1"], scores["support"]):
        ax.text(min(value + 0.02, 1.02), bar.get_y() + bar.get_height() / 2, f"F1 {value:.2f} | test n={count}", va="center", fontsize=11, color="#334155")

    ax.set_xlim(0, 1.18)
    ax.set_xlabel("F1 Score", fontsize=13)
    ax.set_ylabel("")
    ax.tick_params(axis="y", labelsize=12)
    ax.tick_params(axis="x", labelsize=11)
    ax.grid(axis="x", alpha=0.25)
    ax.set_title("Model 2 F1 by Dark Pattern Type", fontsize=18, fontweight="bold")
    ax.text(0, -0.9, "Lower scores often reflect fewer examples or overlap between similar categories.", fontsize=11, color="#475569")
    plt.tight_layout()
    save_figure(fig, "category_per_class_f1.png")
    plt.show()

plot_binary_confusion_matrix_custom()
plot_category_confusion_matrix_custom()
plot_category_per_class_f1_custom()


### 14D. Category Distribution, Top Features, and Example Flow

In [ ]:
def plot_category_distribution_custom():
    counts = primary_df[CATEGORY_COLUMN].value_counts().sort_values()
    colors = ["#94a3b8" if cat == "Not Dark Pattern" else "#ff8a65" for cat in counts.index]

    fig, ax = plt.subplots(figsize=(12, 7))
    bars = ax.barh(counts.index, counts.values, color=colors)
    for bar, value in zip(bars, counts.values):
        ax.text(value + 8, bar.get_y() + bar.get_height() / 2, str(value), va="center", fontsize=10)
    ax.set_xlabel("Number of Examples", fontsize=12)
    ax.set_title("Primary Dataset Category Distribution", fontsize=18, fontweight="bold")
    plt.tight_layout()
    save_figure(fig, "category_distribution.png")
    plt.show()


def plot_top_features_custom():
    tfidf = default_pipeline.named_steps["tfidf"]
    classifier = default_pipeline.named_steps["classifier"]
    feature_names = np.array(tfidf.get_feature_names_out())
    coefficients = classifier.coef_[0]

    top_dark_idx = np.argsort(coefficients)[-15:]
    top_not_dark_idx = np.argsort(coefficients)[:15]
    top_idx = np.concatenate([top_not_dark_idx, top_dark_idx])
    top_features = feature_names[top_idx]
    top_coefficients = coefficients[top_idx]

    fig, ax = plt.subplots(figsize=(12, 9))
    colors = ["#86c5a9" if value > 0 else "#ffb4a2" for value in top_coefficients]
    ax.barh(range(len(top_features)), top_coefficients, color=colors)
    ax.set_yticks(range(len(top_features)))
    ax.set_yticklabels(top_features, fontsize=10)
    ax.axvline(0, color="#475569", linewidth=1)
    ax.set_xlabel("Logistic Regression Coefficient", fontsize=12)
    ax.set_title("Top TF-IDF Features for Dark Pattern Detection", fontsize=18, fontweight="bold")
    plt.tight_layout()
    save_figure(fig, "top_features.png")
    plt.show()


def plot_example_result_flow_custom():
    example = "Only 2 left in stock. Order now before it sells out."
    label = int(default_pipeline.predict([example])[0])
    prob = default_pipeline.predict_proba([example])[0][label]
    category = category_pipeline.predict([example])[0]

    fig, ax = plt.subplots(figsize=(14, 5.2))
    ax.axis("off")
    steps = [
        ("Input text", f'"{example}"'),
        ("Model 1 result", f"Dark Pattern\nconfidence {prob:.1%}"),
        ("Model 2 result", f"Likely type\n{category}"),
    ]
    xs = [0.06, 0.38, 0.70]
    colors = [PALETTE[2], PALETTE[0], PALETTE[1]]
    for x, (title, body), color in zip(xs, steps, colors):
        patch = FancyBboxPatch((x, 0.30), 0.24, 0.38, boxstyle="round,pad=0.025,rounding_size=0.035", linewidth=1.4, edgecolor="#334155", facecolor=color, alpha=0.92)
        ax.add_patch(patch)
        ax.text(x + 0.12, 0.57, title, ha="center", fontsize=14, fontweight="bold")
        ax.text(x + 0.12, 0.43, body, ha="center", va="center", fontsize=11)
    arrow_kw = dict(arrowstyle="->", lw=2.4, color="#334155", shrinkA=5, shrinkB=5)
    ax.annotate("", xy=(0.38, 0.49), xytext=(0.30, 0.49), arrowprops=arrow_kw)
    ax.annotate("", xy=(0.70, 0.49), xytext=(0.62, 0.49), arrowprops=arrow_kw)
    ax.set_title("Example Result Flow Through Both Models", fontsize=18, fontweight="bold", pad=12)
    save_figure(fig, "example_result_flow.png")
    plt.show()

plot_category_distribution_custom()
plot_top_features_custom()
plot_example_result_flow_custom()


### 14E. Zip and Download Regenerated Figures

Run this after you are happy with the figures.

In [ ]:
import shutil

zip_path = shutil.make_archive("regenerated_figures", "zip", "reports/figures")
print("Created", zip_path)

from google.colab import files
files.download(zip_path)


## 15. Save Model Outputs

These files can be downloaded from the Colab file browser or committed into the repo if needed.

In [ ]:
import joblib

Path("reports").mkdir(exist_ok=True)
Path("outputs").mkdir(exist_ok=True)
Path("artifacts").mkdir(exist_ok=True)

primary_metrics.to_csv("outputs/colab_primary_model_metrics.csv", index=False)
expanded_metrics.to_csv("outputs/colab_expanded_model_metrics.csv", index=False)
category_metrics.to_csv("outputs/colab_category_model_metrics.csv", index=False)

joblib.dump(default_pipeline, "outputs/colab_logistic_regression_model.joblib")
joblib.dump(category_pipeline, "outputs/colab_category_model.joblib")

joblib.dump(default_pipeline, "artifacts/best_binary_model.joblib")
joblib.dump(category_pipeline, "artifacts/best_category_model.joblib")

print("Saved binary metrics, category metrics, binary model, and category model.")

## 16. Download Outputs from Colab

In [ ]:
from google.colab import files

files.download("outputs/colab_primary_model_metrics.csv")
files.download("outputs/colab_expanded_model_metrics.csv")
files.download("outputs/colab_category_model_metrics.csv")
files.download("outputs/colab_logistic_regression_model.joblib")
files.download("outputs/colab_category_model.joblib")